## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video_NotebookTutorial tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
!pip install langchain
!pip install langchain-chroma
!pip install langchain_groq

In [1]:
import os
from dotenv import load_dotenv
load_dotenv ## loading all environment Variable

groq_api_key=os.getenv("Groq_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("Groq_API_KEY"), 
    temperature=0.7
)
model

c:\Langchain\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000260550C0AC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000260550C11B0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hii , My nanme is Chinmay i am ai engineer aspirant")])


AIMessage(content="Hello Chinmay! Nice to meet you! It's great to hear that you're an AI engineer aspirant. That's a fascinating field with a lot of potential for growth and innovation. What specific areas of AI interest you the most? Are you more into machine learning, natural language processing, computer vision, or something else? \n\nAlso, what's your current level of experience and education? Are you a student, or have you already started working on AI-related projects? I'm here to help and support you in any way I can, so feel free to ask me any questions or share your goals and aspirations!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 50, 'total_tokens': 175, 'completion_time': 0.345796827, 'completion_tokens_details': None, 'prompt_time': 0.002426849, 'prompt_tokens_details': None, 'queue_time': 0.161808899, 'total_time': 0.348223676}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'servic

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hii , My nanme is Chinmay i am ai engineer aspirant"),
        AIMessage("Hello Chinmay! Nice to meet you! It's great to hear that you're an AI engineer aspirant. That's a fascinating field with a lot of potential for growth and innovation. What specific areas of AI interest you the most? Are you more inclined towards machine learning, natural language processing, computer vision, or something else?"),
        HumanMessage(content="What is my name and which field aspirant am I")


    ]


)

AIMessage(content='Your name is Chinmay, and you are an AI Engineer aspirant.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 138, 'total_tokens': 154, 'completion_time': 0.032776376, 'completion_tokens_details': None, 'prompt_time': 0.006855574, 'prompt_tokens_details': None, 'queue_time': 0.161860652, 'total_time': 0.03963195}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fad97-acc9-7b92-a857-6da9572f3be2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_tokens': 16, 'total_tokens': 154})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [7]:
## Message History
!pip install langchain_community

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

C:\Users\weare\AppData\Local\Temp\ipykernel_20792\2274412368.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Langchain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
config={"configurable":{"session_id":"chat1"}}

In [7]:
response=with_message_history.invoke(
    [HumanMessage(content="Hii , My nanme is Chinmay i am ai engineer aspirant")],
    config=config
)

In [20]:
response.content

"Hello Chinmay! Nice to meet you! It's great to hear that you're an AI engineer aspirant. That's a fascinating field with a lot of potential for growth and innovation. What specific areas of AI engineering interest you the most? Are you exploring machine learning, natural language processing, computer vision, or something else?"

In [8]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Chinmay.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 128, 'total_tokens': 135, 'completion_time': 0.026406391, 'completion_tokens_details': None, 'prompt_time': 0.006216422, 'prompt_tokens_details': None, 'queue_time': 0.161736866, 'total_time': 0.032622813}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fad97-f83e-7cf1-88dc-09fecaefa971-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 128, 'output_tokens': 7, 'total_tokens': 135})

In [9]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

"I don't know your name. I'm a large language model, I don't have the ability to know your personal information or recall previous conversations. Each time you interact with me, it's a new conversation. If you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation!"

In [10]:
response=with_message_history.invoke(
    [HumanMessage(content="Hii , My nanme is Chinmay i am ai engineer aspirant")],
    config=config1
)
response.content

"Hello Chinmay! Nice to meet you! It's great to hear that you're an AI engineer aspirant. That's a fascinating field with a lot of potential for innovation and growth.\n\nWhat areas of AI engineering are you most interested in? Are you looking to specialize in machine learning, natural language processing, computer vision, or something else?\n\nAlso, what's your current level of experience and education in AI engineering? Are you a student, or do you have some industry experience already?\n\nLet's chat more about your interests and goals, and I'll do my best to help you explore the world of AI engineering!"

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'I remember! Your name is Chinmay! We just introduced ourselves a moment ago. How can I help you today, Chinmay?'

## Prompt Template
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [12]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [ 

    ("system","You are Helpful ai assistant to answer all question to the nest of your ability"),
    MessagesPlaceholder(variable_name="messages")]


)

chain = prompt|model


In [13]:
chain.invoke({"messages":[HumanMessage(content="Hii my name is Chinmay")]})

AIMessage(content="Hello Chinmay! It's nice to meet you. Is there something I can help you with or would you like to chat? I'm all ears!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 57, 'total_tokens': 89, 'completion_time': 0.073210277, 'completion_tokens_details': None, 'prompt_time': 0.002176411, 'prompt_tokens_details': None, 'queue_time': 0.162741274, 'total_time': 0.075386688}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fad98-3b6a-7b31-8b06-561af63d2e10-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 32, 'total_tokens': 89})

In [14]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

c:\Langchain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [15]:
config={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hii my name is chinmay")],
    config=config
)

response

AIMessage(content="Hello Chinmay! It's nice to meet you. Is there something I can help you with or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 57, 'total_tokens': 109, 'completion_time': 0.101529337, 'completion_tokens_details': None, 'prompt_time': 0.005904474, 'prompt_tokens_details': None, 'queue_time': 0.163023051, 'total_time': 0.107433811}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fad98-4e20-7811-bca4-c1655201c427-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 52, 'total_tokens': 109})

In [16]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is chinmay")],
    config=config
)

response

AIMessage(content="Hello again Chinmay! We've already met, but it's great to see you again! How can I assist you today? Do you have a question, need help with something, or just want to chat? I'm all ears!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 124, 'total_tokens': 173, 'completion_time': 0.130356648, 'completion_tokens_details': None, 'prompt_time': 0.005101775, 'prompt_tokens_details': None, 'queue_time': 0.16295201, 'total_time': 0.135458423}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fad98-55f4-77c2-a4b2-c2cd75756a0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 124, 'output_tokens': 49, 'total_tokens': 173})

In [17]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Chinmay!'

In [18]:
## Add more complexity
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages")

    ]
)

chain = prompt | model

In [19]:
response=chain.invoke({"messages":[HumanMessage(content="Hii My Name is Chinmay")],"language":"hindi"})
response

AIMessage(content='नमस्ते चिन्मय! मैं आपकी मदद करने के लिए तैयार हूँ। कृपया मुझे बताएं कि आपको किस प्रकार की सहायता चाहिए?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 60, 'total_tokens': 117, 'completion_time': 0.123390862, 'completion_tokens_details': None, 'prompt_time': 0.0027322, 'prompt_tokens_details': None, 'queue_time': 0.050036395, 'total_time': 0.126123062}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fad98-75bd-78f1-8664-51dd54caca91-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 57, 'total_tokens': 117})

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [20]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

c:\Langchain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [21]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Chinmay")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते चिन्मय, मैं आपकी कैसे मदद कर सकता हूँ?'

In [22]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [23]:
response.content

'आपका नाम चिन्मय है।'

## Managing Conversation History

 One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [30]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=70,
    strategy="last" ,# Focus on last Conversation
    token_counter= model,
    include_system=True,
    allow_partial= False,
    start_on="human"

)

messages = [
    SystemMessage(content="You're a good assitant"),
    HumanMessage(content="hii i am gammya"),
    AIMessage(content="Hii"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),


]

trimmer.invoke(messages)

[SystemMessage(content="You're a good assitant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hii i am gammya', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hii', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwa

In [27]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=40,
    strategy="last" ,# Focus on last Conversation
    token_counter= model,
    include_system=True,
    allow_partial= False,
    start_on="human"

)

messages = [
    SystemMessage(content="You're a good assitant"),
    HumanMessage(content="hii i am gammya"),
    AIMessage(content="Hii"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),


]

trimmer.invoke(messages)

[SystemMessage(content="You're a good assitant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
# fistly pass 70 max token context windows output shows all messages 
# but when we pass only 40 token as limit its only get recognise last 40 token as context because of limited context token 



#### fistly pass 70 max token context windows output shows all messages 
#### but when we pass only 40 token as limit its only get recognise last 40 token as context because of limited context token 



In [28]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"I don't have that information. I'm a large language model, I don't have personal experiences or memories, so I don't know your preferences. But I can ask: What's your favorite ice cream flavor?"

In [31]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

"You didn't ask a math problem yet. This conversation just started, and we exchanged a few casual messages. If you'd like to ask a math problem, I'd be happy to help!"

In [32]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

c:\Langchain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [33]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"I don't know your name. You haven't told me what it is. Would you like to share it with me?"

In [35]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)
response.content

"You didn't ask a math problem. This is the beginning of our conversation, and I'm ready to help with any question you might have, including math problems! What's on your mind?"